%md
# Python using linting

## Linting context
- linting can become frustrating until habits form
- linting can impress rules we dont desire
- linting can create work that does not bring value

## Making good use of linting
- change the rules to match the project needs see toml (we could also git ignore an rcfile if we want personal rules potentially)
- turn up or down the sensativity such that it brings warning that bring value without creating busy work


# Python linting use
- For now we can run this locally before a push
- We can look at it as part of a PR
- For now if it improves the code, gets us in good habits and doesnt slow us much then its a good tool therefore
  - just stick the output into AI for now and get it to make some recommendation and decide if you think any changes should be made to the lint rules or the code. For example: "Function table_loader is missing a 'spark' argument" AI will tell you that though spark is provided by databricks by making it an argument you make the function unit testable (this is because github environment doesnt have spark as default so we pass it in, and also running in databrick we will want to pass in or mock spark).
  - we probably should develop some rules of thumb, or simple queston of what we do with linting suggestions something like
    - Does it help with unit testing
    - Does it make it more reuseable
    - Is it easy to implement
    - Is it an impotant suggestion or nicety

# In Future
- in future we would want code reports
- in future we would want to block pushes if coverage percentage goes down

# Install for lint

In [0]:
# Databricks notebook source
# MAGIC %md


%pip install databricks-labs-pylint -q
dbutils.library.restartPython()




# Import for Lint

In [0]:
import sys
import subprocess
import json
import re

# This code allows the testing of specific files, so you can focus on the code just added

Its useful too to check with the new code
- can it be refactored into shareable functions
- can it use a previously created function
- can it be made unit testable
- has a unit test been created for it

In [0]:

dbutils.widgets.text("file_path", "/Workspace/Users/philip.tate@nhs.net/PT Separate Feature Branch/src/ingestion/ods_ingest.py", "📁 File Path to Lint")

file_path = dbutils.widgets.get("file_path")
# Toml provides load-plugins=databricks.labs.pylint.all
# Toml provides lint rules
cmd = [
    sys.executable, 
    '-m', 
    'pylint', 
    file_path
    ]
#    "--rcfile=/Workspace/Users/philip.tate@nhs.net/PT Separate Feature Branch/pyproject.toml", # works but its picking up the toml now
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
print("-----")
lines = result.stdout.strip().split('\n')
print(f"Last line: {lines[-1] if lines else 'EMPTY'}")

last_line = lines[-1] if lines else ""
score_match = re.search(r"([0-9.-]+)/10", last_line)
if score_match:
    score = float(score_match.group(1))
else:
    score = None
print(f"Score: {score}")
print("-----")
print(result.stderr)


# This is pseudo code currently !!!!!! (But it does work enough. Files disapear when they are 0.00 because no error entries so will generally see 5.0 until disapears)
- the change is what is interesting so actually we should not parse the numbers at all and should just print the last line per file.

# Lint Code Quality Report (For future use)

It does not make sense to be correcting code that is not part of task, if that is required it should be a task in its self. However it is useful to see the direction the code base is going.
If the code score gets worse when a feature is added we can in future have cicd block the merge and require improvement. This means little by little it would improve.

The report is visible in test/test-reports.

# Future
Would be good to get package working like coverlet, and cicd gitpage reports like TELBlazor, out of scope for now.


In [0]:
import sys
import subprocess
import json
import re
from pathlib import Path
from html import escape
from datetime import datetime

# Paths
root_path = "/Workspace/Users/philip.tate@nhs.net/PT Separate Feature Branch/src"
report_folder = Path("test-reports")
report_folder.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_path = report_folder / f"pylint_report_{timestamp}.html"


def get_score_from_stdout(stdout_text):
    """Parse score from pylint stdout text output"""
    if not stdout_text:
        print("  Error: stdout_text is empty")
        return None
    lines = stdout_text.strip().split('\n')
    last_line = lines[-1] if lines else ""
    print(f"  Last line: {last_line}")
    
    score_match = re.search(r"([0-9.-]+)/10", last_line)
    if score_match:
        score_str = score_match.group(1)
        score = float(score_str)
        #print(f"  Parsed score string: '{score_str}' -> float: {score}")
        return score
    else:
        print(f"  No score found")
        return None


def get_issues_json(file_path):
    """Get JSON issues for a file"""
    cmd = [sys.executable, "-m", "pylint", file_path, "--output-format=json"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    try:
        issues = json.loads(result.stdout)
        print(f"  Found {len(issues)} issues")
        return issues
    except json.JSONDecodeError:
        print(f"  Failed to parse JSON")
        return []


def run_pylint_on_src():
    """Run pylint on src directory, return aggregate score and file list"""
    print("Running pylint on src directory...")
    
    # Get text output for aggregate score
    cmd_text = [sys.executable, "-m", "pylint", root_path]
    result_text = subprocess.run(cmd_text, capture_output=True, text=True)
    
    print("\n=== SRC TEXT OUTPUT (last 500 chars) ===")
    print(result_text.stdout[-500:])
    print("=========================================\n")
    
    aggregate_score = get_score_from_stdout(result_text.stdout)
    print(f"Aggregate score: {aggregate_score}\n")
    
    # Get JSON output for file list
    cmd_json = [sys.executable, "-m", "pylint", root_path, "--output-format=json"]
    result_json = subprocess.run(cmd_json, capture_output=True, text=True)
    
    try:
        all_issues = json.loads(result_json.stdout)
        py_files = sorted(set(issue['path'] for issue in all_issues))
       #print(f"Found {len(py_files)} files from JSON\n")
        return aggregate_score, py_files
    except json.JSONDecodeError:
        print("Failed to parse JSON for file list\n")
        return aggregate_score, []


# Main execution
aggregate_score, py_files = run_pylint_on_src()

# Process each file
file_data = {}
#print("Processing individual files...")

for py_file in py_files:
    #print(f"\n{Path(py_file).name}:")
    #print(f"\nProcessing: {py_file}")
    # Get score
    cmd = [sys.executable, "-m", "pylint", py_file]
    result = subprocess.run(cmd, capture_output=True, text=True)
    score = get_score_from_stdout(result.stdout)
    
    # Get issues
    issues = get_issues_json(py_file)
    
    file_data[py_file] = {'score': score, 'issues': issues}

# Build HTML
html = f"""
<html>
<head><title>Pylint Report</title>
<style>
body {{ font-family: Arial, sans-serif; }}
h1 {{ text-align: center; }}
h2 {{ margin-top: 40px; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #999; padding: 4px; text-align: left; }}
th {{ background: #eee; }}
tr:nth-child(even) {{ background: #f9f9f9; }}
</style>
</head>
<body>
<h1>Pylint Aggregate Score: {aggregate_score if aggregate_score is not None else "NOT FOUND"}</h1>
"""

for py_file, data in file_data.items():
    score_display = data['score'] if data['score'] is not None else "NOT FOUND"
    #html += f"<h2>{Path(py_file).name} — Score: {score_display}</h2>"
    html += f"<h2>File: {py_file} — Score: {score_display}</h2>"
    html += "<table><tr><th>Line</th><th>Code</th><th>Message</th></tr>"
    for issue in data['issues']:
        html += f"<tr><td>{issue.get('line','')}</td>"
        html += f"<td>{escape(issue.get('message-id',''))}</td>"
        html += f"<td>{escape(issue.get('message',''))}</td></tr>"
    html += "</table>"

html += "</body></html>"

with open(html_path, "w") as f:
    f.write(html)

print(f"\n\nPylint HTML report written to: {html_path}")
displayHTML(html_path.read_text())

# Note : only seems to be detecting .py
Below is using the library for dbx nb files


is only possible through client?

https://github.com/databrickslabs/pylint-plugin?tab=readme-ov-file#integration-with-databricks-cli